# LeetCode 98: Validate Binary Search Tree

**Difficulty**: Medium  
**Topics**: Tree, Binary Search Tree, Depth-First Search, Recursion  
**Link**: [LeetCode Problem](https://leetcode.com/problems/validate-binary-search-tree/)

---

## Problem Statement

Given the `root` of a binary tree, determine if it is a valid binary search tree (BST).

A **valid BST** is defined as follows:
- The left subtree of a node contains only nodes with keys **less than** the node's key.
- The right subtree of a node contains only nodes with keys **greater than** the node's key.
- Both the left and right subtrees must also be binary search trees.

### Examples

**Example 1:**
```
    2
   / \
  1   3

Input: root = [2,1,3]
Output: true
```

**Example 2:**
```
    5
   / \
  1   4
     / \
    3   6

Input: root = [5,1,4,null,null,3,6]
Output: false
Explanation: The root node's value is 5 but its right child's value is 4.
```

### Constraints

- The number of nodes in the tree is in the range `[1, 10^4]`.
- `-2^31 <= Node.val <= 2^31 - 1`

---

## Understanding BST Properties

### Common Misconception

**WRONG**: Just check if `left.val < node.val < right.val`

```
Example that breaks this:
      10
     /  \
    5    15
        /  \
       6   20

At node 15: left(6) < 15 < right(20) ✓
But 6 < 10, so 6 should be in left subtree of 10!
This is NOT a valid BST.
```

### Correct Understanding

**Every node must satisfy range constraints:**

```
For a node with value V:
- ALL nodes in left subtree must be < V
- ALL nodes in right subtree must be > V
```

### Range Propagation

```
Example: Valid BST
        10 (range: -∞ to +∞)
       /  \
      5    15 (range: 10 to +∞)
     / \   / \
    2   7 12  20

Node 10: must be in (-∞, +∞)
Node 5:  must be in (-∞, 10)  [left of 10]
Node 15: must be in (10, +∞)  [right of 10]
Node 2:  must be in (-∞, 5)   [left of 5]
Node 7:  must be in (5, 10)   [right of 5, but still left of 10]
Node 12: must be in (10, 15)  [left of 15, but still right of 10]
Node 20: must be in (15, +∞)  [right of 15]
```

### Key Insight

As we traverse:
- Going **left**: upper bound becomes current node's value
- Going **right**: lower bound becomes current node's value

---

## Tree Node Definition

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """
    Build tree from level-order list.
    None represents null nodes.
    """
    if not values:
        return None
    
    root = TreeNode(values[0])
    queue = [root]
    i = 1
    
    while queue and i < len(values):
        node = queue.pop(0)
        
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    
    return root

def print_tree(root, prefix="", is_tail=True):
    """
    Print tree structure.
    """
    if root is None:
        return
    
    print(prefix + ("└── " if is_tail else "├── ") + str(root.val))
    
    children = []
    if root.left is not None:
        children.append((root.left, False))
    if root.right is not None:
        children.append((root.right, True))
    
    for i, (child, is_last) in enumerate(children):
        if i == len(children) - 1:
            is_last = True
        extension = "    " if is_tail else "│   "
        print_tree(child, prefix + extension, is_last)

---

## Approach 1: Recursive with Range Validation

### Algorithm

1. Start with root having range `(-∞, +∞)`
2. For each node:
   - Check if node value is within valid range
   - Recursively validate left subtree with range `(min, node.val)`
   - Recursively validate right subtree with range `(node.val, max)`
3. Return true if all nodes satisfy their range constraints

### Complexity

- **Time**: O(n) - visit each node once
- **Space**: O(h) - recursion stack, where h is tree height
  - Best case (balanced): O(log n)
  - Worst case (skewed): O(n)

### Why This Works

By passing down range constraints, we ensure:
- Every node in left subtree is less than all ancestors on the path
- Every node in right subtree is greater than all ancestors on the path

In [ ]:
def isValidBST_recursive(root):
    """
    Validate BST using recursive range checking.
    Time: O(n)
    Space: O(h)
    """
    def validate(node, min_val, max_val):
        # Empty tree is valid
        if not node:
            return True
        
        # Check if current node violates range
        if node.val <= min_val or node.val >= max_val:
            return False
        
        # Recursively validate left and right subtrees
        # Left: all values must be < node.val
        # Right: all values must be > node.val
        return (validate(node.left, min_val, node.val) and
                validate(node.right, node.val, max_val))
    
    return validate(root, float('-inf'), float('inf'))

# Test cases
print("Approach 1: Recursive Range Validation\n")

# Valid BST
tree1 = build_tree([2, 1, 3])
print("Tree 1:")
print_tree(tree1)
print(f"Valid BST: {isValidBST_recursive(tree1)}\n")

# Invalid BST
tree2 = build_tree([5, 1, 4, None, None, 3, 6])
print("Tree 2:")
print_tree(tree2)
print(f"Valid BST: {isValidBST_recursive(tree2)}\n")

# Tricky case: left child equals parent
tree3 = build_tree([1, 1])
print("Tree 3:")
print_tree(tree3)
print(f"Valid BST: {isValidBST_recursive(tree3)}")

## Detailed Recursive Trace

In [ ]:
def isValidBST_verbose(root):
    """
    Verbose version showing recursive calls and range checks.
    """
    def validate(node, min_val, max_val, indent=0):
        prefix = "  " * indent
        
        if not node:
            print(f"{prefix}→ Node: None, Range: ({min_val}, {max_val}) → Valid (empty)")
            return True
        
        print(f"{prefix}→ Node: {node.val}, Range: ({min_val}, {max_val})")
        
        # Check range
        if node.val <= min_val or node.val >= max_val:
            print(f"{prefix}  ✗ INVALID: {node.val} not in ({min_val}, {max_val})")
            return False
        
        print(f"{prefix}  ✓ {node.val} is in ({min_val}, {max_val})")
        
        # Validate left subtree
        print(f"{prefix}  Checking left subtree (range: {min_val}, {node.val}):")
        left_valid = validate(node.left, min_val, node.val, indent + 2)
        
        # Validate right subtree
        print(f"{prefix}  Checking right subtree (range: {node.val}, {max_val}):")
        right_valid = validate(node.right, node.val, max_val, indent + 2)
        
        result = left_valid and right_valid
        print(f"{prefix}  Node {node.val}: {'Valid' if result else 'Invalid'}")
        return result
    
    print("Validating BST with detailed trace:\n")
    print("="*70)
    result = validate(root, float('-inf'), float('inf'))
    print("="*70)
    print(f"\nFinal Result: {'Valid BST' if result else 'Invalid BST'}")
    return result

# Trace valid BST
print("Example 1: Valid BST\n")
tree = build_tree([10, 5, 15, 2, 7, 12, 20])
print("Tree structure:")
print_tree(tree)
print()
isValidBST_verbose(tree)

print("\n" + "="*70 + "\n")

# Trace invalid BST
print("Example 2: Invalid BST\n")
tree = build_tree([10, 5, 15, None, None, 6, 20])
print("Tree structure:")
print_tree(tree)
print()
isValidBST_verbose(tree)

---

## Approach 2: Inorder Traversal (Recursive)

### Key Insight

**Inorder traversal of a BST produces values in ascending order!**

```
Valid BST:
    10
   /  \
  5    15
 / \   / \
2   7 12  20

Inorder: 2, 5, 7, 10, 12, 15, 20 ✓ (strictly increasing)

Invalid BST:
    10
   /  \
  5    15
      /  \
     6   20

Inorder: 5, 10, 6, 15, 20 ✗ (10 > 6, not increasing)
```

### Algorithm

1. Perform inorder traversal (left → root → right)
2. Track previous node value
3. If current value ≤ previous value, return false
4. If all values are strictly increasing, return true

### Complexity

- **Time**: O(n) - visit each node once
- **Space**: O(h) - recursion stack

In [ ]:
def isValidBST_inorder_recursive(root):
    """
    Validate BST using inorder traversal (recursive).
    Time: O(n)
    Space: O(h)
    """
    prev = [None]  # Use list to allow modification in nested function
    
    def inorder(node):
        if not node:
            return True
        
        # Check left subtree
        if not inorder(node.left):
            return False
        
        # Check current node
        if prev[0] is not None and node.val <= prev[0]:
            return False
        prev[0] = node.val
        
        # Check right subtree
        return inorder(node.right)
    
    return inorder(root)

# Test
print("Approach 2: Inorder Traversal (Recursive)\n")

tree1 = build_tree([2, 1, 3])
print("Tree 1:")
print_tree(tree1)
print(f"Valid BST: {isValidBST_inorder_recursive(tree1)}\n")

tree2 = build_tree([5, 1, 4, None, None, 3, 6])
print("Tree 2:")
print_tree(tree2)
print(f"Valid BST: {isValidBST_inorder_recursive(tree2)}")

## Inorder Traversal Trace

In [ ]:
def isValidBST_inorder_verbose(root):
    """
    Inorder traversal with detailed trace.
    """
    prev = [None]
    sequence = []
    
    def inorder(node, indent=0):
        prefix = "  " * indent
        
        if not node:
            print(f"{prefix}→ None")
            return True
        
        print(f"{prefix}→ Visit node {node.val}")
        
        # Left
        print(f"{prefix}  Go left:")
        if not inorder(node.left, indent + 2):
            return False
        
        # Current
        print(f"{prefix}  Process {node.val}:")
        if prev[0] is not None:
            print(f"{prefix}    Compare: prev={prev[0]}, current={node.val}")
            if node.val <= prev[0]:
                print(f"{prefix}    ✗ INVALID: {node.val} <= {prev[0]} (not strictly increasing)")
                return False
            print(f"{prefix}    ✓ Valid: {prev[0]} < {node.val}")
        else:
            print(f"{prefix}    First node in traversal")
        
        sequence.append(node.val)
        prev[0] = node.val
        
        # Right
        print(f"{prefix}  Go right:")
        return inorder(node.right, indent + 2)
    
    print("Inorder Traversal Validation:\n")
    print("="*70)
    result = inorder(root)
    print("="*70)
    print(f"\nInorder sequence: {sequence}")
    print(f"Strictly increasing: {result}")
    print(f"Result: {'Valid BST' if result else 'Invalid BST'}")
    return result

# Trace
print("Example: Valid BST\n")
tree = build_tree([10, 5, 15, 2, 7])
print("Tree:")
print_tree(tree)
print()
isValidBST_inorder_verbose(tree)

print("\n" + "="*70 + "\n")

print("Example: Invalid BST\n")
tree = build_tree([10, 5, 15, None, None, 6, 20])
print("Tree:")
print_tree(tree)
print()
isValidBST_inorder_verbose(tree)

---

## Approach 3: Inorder Traversal (Iterative)

### Algorithm

Use a stack to simulate recursion:
1. Push all left nodes onto stack
2. Pop node, check if value > previous
3. Move to right child
4. Repeat until stack is empty

### Complexity

- **Time**: O(n)
- **Space**: O(h) - stack size

### Advantages

- No recursion overhead
- Can stop early on first violation
- More control over traversal

In [ ]:
def isValidBST_inorder_iterative(root):
    """
    Validate BST using iterative inorder traversal.
    Time: O(n)
    Space: O(h)
    """
    stack = []
    prev = None
    current = root
    
    while stack or current:
        # Go to leftmost node
        while current:
            stack.append(current)
            current = current.left
        
        # Process node
        current = stack.pop()
        
        # Check if strictly increasing
        if prev is not None and current.val <= prev:
            return False
        
        prev = current.val
        
        # Move to right subtree
        current = current.right
    
    return True

# Test
print("Approach 3: Inorder Traversal (Iterative)\n")

tree1 = build_tree([2, 1, 3])
print("Tree 1:")
print_tree(tree1)
print(f"Valid BST: {isValidBST_inorder_iterative(tree1)}\n")

tree2 = build_tree([5, 1, 4, None, None, 3, 6])
print("Tree 2:")
print_tree(tree2)
print(f"Valid BST: {isValidBST_inorder_iterative(tree2)}")

## Iterative Inorder Trace

In [ ]:
def isValidBST_iterative_verbose(root):
    """
    Iterative inorder with detailed trace.
    """
    stack = []
    prev = None
    current = root
    sequence = []
    step = 0
    
    print("Iterative Inorder Traversal:\n")
    print(f"{'Step':<6} {'Action':<30} {'Stack':<20} {'Prev':<8} {'Current'}")
    print("="*80)
    
    while stack or current:
        # Go to leftmost node
        while current:
            step += 1
            stack.append(current)
            stack_vals = [str(n.val) for n in stack]
            print(f"{step:<6} Push {current.val}, go left {' '*13} {stack_vals} {prev if prev else 'None':<8} {current.val if current else 'None'}")
            current = current.left
        
        # Process node
        step += 1
        current = stack.pop()
        stack_vals = [str(n.val) for n in stack]
        
        print(f"{step:<6} Pop {current.val} {' '*22} {stack_vals if stack_vals else '[]'} {prev if prev else 'None':<8} {current.val}")
        
        # Check
        if prev is not None and current.val <= prev:
            print(f"\n✗ INVALID: {current.val} <= {prev} (not strictly increasing)")
            print(f"Sequence so far: {sequence}")
            return False
        
        if prev is not None:
            print(f"       ✓ {prev} < {current.val}")
        
        sequence.append(current.val)
        prev = current.val
        
        # Move to right
        current = current.right
    
    print("="*80)
    print(f"\nInorder sequence: {sequence}")
    print(f"Result: Valid BST")
    return True

# Trace
tree = build_tree([10, 5, 15, 2, 7])
print("Tree:")
print_tree(tree)
print()
isValidBST_iterative_verbose(tree)

---

## Edge Cases and Special Scenarios

In [ ]:
print("Edge Cases Testing\n")
print("="*70)

# Edge Case 1: Single node
print("\nCase 1: Single node")
tree = build_tree([1])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: Single node is always a valid BST\n")

# Edge Case 2: Two nodes - left child
print("Case 2: Two nodes - left child")
tree = build_tree([2, 1])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: 1 < 2, valid\n")

# Edge Case 3: Two nodes - right child
print("Case 3: Two nodes - right child")
tree = build_tree([1, None, 2])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: 2 > 1, valid\n")

# Edge Case 4: Duplicate values (left)
print("Case 4: Duplicate values - left child equals parent")
tree = build_tree([1, 1])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: BST requires STRICTLY less than, 1 <= 1 is invalid\n")

# Edge Case 5: Duplicate values (right)
print("Case 5: Duplicate values - right child equals parent")
tree = build_tree([1, None, 1])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: BST requires STRICTLY greater than, 1 >= 1 is invalid\n")

# Edge Case 6: Integer limits
print("Case 6: Integer limits")
tree = build_tree([2147483647])  # INT_MAX
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: Single node with max value is valid\n")

# Edge Case 7: Negative values
print("Case 7: Negative values")
tree = build_tree([0, -1, 1])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: -1 < 0 < 1, valid\n")

# Edge Case 8: All left skewed
print("Case 8: Left-skewed tree")
tree = build_tree([5, 4, None, 3, None, 2, None, 1])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: 1 < 2 < 3 < 4 < 5, valid\n")

# Edge Case 9: All right skewed
print("Case 9: Right-skewed tree")
tree = build_tree([1, None, 2, None, 3, None, 4, None, 5])
print_tree(tree)
print(f"Valid: {isValidBST_recursive(tree)}")
print("Explanation: 1 < 2 < 3 < 4 < 5, valid")

## Tricky Test Cases

In [ ]:
print("Tricky Test Cases\n")
print("="*70)

# Tricky 1: Looks valid locally but invalid globally
print("\nTricky Case 1: Locally valid, globally invalid")
tree = build_tree([10, 5, 15, None, None, 6, 20])
print("Tree:")
print_tree(tree)
print(f"\nValid: {isValidBST_recursive(tree)}")
print("Why invalid: Node 6 is in right subtree of 10")
print("             But 6 < 10, should be in left subtree!")
print("             At node 15: left(6) < 15 ✓ locally")
print("             But 6 must be > 10 (ancestor) ✗ globally\n")

# Tricky 2: Deep violation
print("Tricky Case 2: Deep violation")
tree = build_tree([10, 5, 15, 2, 7, 12, 20, None, None, None, 8])
print("Tree:")
print_tree(tree)
print(f"\nValid: {isValidBST_recursive(tree)}")
print("Why invalid: Node 8 is in right subtree of 7")
print("             8 > 7 ✓ locally")
print("             But 8 must be < 10 (ancestor) and 8 < 10 ✓")
print("             Wait, this should be valid! Let me check...")
print("             Actually, 8 is in left subtree of 10, so 8 < 10 ✓")
print("             8 is in right subtree of 7, so 8 > 7 ✓")
print("             This IS valid!\n")

# Tricky 3: Boundary violation
print("Tricky Case 3: Subtle boundary violation")
tree = build_tree([5, 4, 6, None, None, 3, 7])
print("Tree:")
print_tree(tree)
print(f"\nValid: {isValidBST_recursive(tree)}")
print("Why invalid: Node 3 is in left subtree of 6")
print("             3 < 6 ✓ locally")
print("             But 3 is in right subtree of 5, so must be > 5")
print("             3 < 5 ✗ globally invalid!\n")

# Tricky 4: Perfect BST
print("Tricky Case 4: Perfect BST")
tree = build_tree([8, 4, 12, 2, 6, 10, 14, 1, 3, 5, 7, 9, 11, 13, 15])
print("Tree:")
print_tree(tree)
print(f"\nValid: {isValidBST_recursive(tree)}")
print("This is a perfect, balanced BST")

---

## Performance Comparison

In [ ]:
import time

def create_balanced_bst(n):
    """
    Create a balanced BST with n nodes.
    """
    def build(start, end):
        if start > end:
            return None
        mid = (start + end) // 2
        node = TreeNode(mid)
        node.left = build(start, mid - 1)
        node.right = build(mid + 1, end)
        return node
    
    return build(1, n)

def benchmark(tree, name):
    """
    Benchmark all approaches.
    """
    results = {}
    
    # Recursive range
    start = time.time()
    result = isValidBST_recursive(tree)
    elapsed = time.time() - start
    results['Recursive Range'] = (result, elapsed)
    
    # Inorder recursive
    start = time.time()
    result = isValidBST_inorder_recursive(tree)
    elapsed = time.time() - start
    results['Inorder Recursive'] = (result, elapsed)
    
    # Inorder iterative
    start = time.time()
    result = isValidBST_inorder_iterative(tree)
    elapsed = time.time() - start
    results['Inorder Iterative'] = (result, elapsed)
    
    return results

# Benchmark
print("Performance Comparison\n")
print(f"{'Tree Size':<12} {'Approach':<20} {'Result':<8} {'Time (seconds)'}")
print("="*70)

for size in [100, 500, 1000, 5000]:
    tree = create_balanced_bst(size)
    results = benchmark(tree, f"Size {size}")
    
    for approach, (result, elapsed) in results.items():
        print(f"{size:<12} {approach:<20} {'Valid' if result else 'Invalid':<8} {elapsed:.6f}")
    print()

print("Observation: All approaches have similar O(n) performance.")
print("Iterative approach has slightly less overhead.")

---

## Comparison of Approaches

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Recursive Range** | O(n) | O(h) | Intuitive, clear logic | Recursion overhead |
| **Inorder Recursive** | O(n) | O(h) | Elegant, uses BST property | Needs prev tracking |
| **Inorder Iterative** | O(n) | O(h) | No recursion, can stop early | More code |

### When to Use Each

- **Recursive Range**: Interview (most intuitive), clear reasoning
- **Inorder Recursive**: When you recognize inorder property
- **Inorder Iterative**: Production code, performance-critical

---

## Key Takeaways

### BST Definition

1. **Left subtree**: ALL nodes < root
2. **Right subtree**: ALL nodes > root
3. **Both subtrees**: Must also be BSTs
4. **Strictly less/greater**: No duplicates allowed

### Common Mistakes

❌ **Only checking immediate children**
```python
# WRONG!
if node.left.val < node.val < node.right.val:
    return True
```

✅ **Must check entire subtrees**
```python
# CORRECT!
validate(node.left, min_val, node.val)
validate(node.right, node.val, max_val)
```

### Range Validation Pattern

1. **Start**: `(-∞, +∞)`
2. **Go left**: `(min, node.val)`
3. **Go right**: `(node.val, max)`
4. **Check**: `min < node.val < max`

### Inorder Property

- **Valid BST** → Inorder is **strictly increasing**
- **Invalid BST** → Inorder has **decreasing or equal** values
- Can validate by checking sequence order

### Edge Cases to Remember

1. **Single node**: Always valid
2. **Duplicates**: Always invalid (BST requires strict inequality)
3. **Integer limits**: Use `float('-inf')` and `float('inf')`
4. **Negative values**: BST works with any comparable values
5. **Deep violations**: Must check entire path from root

### Complexity Analysis

- **Time**: O(n) - must visit every node in worst case
- **Space**: O(h) - recursion/stack depth
  - Balanced tree: O(log n)
  - Skewed tree: O(n)

### Remember

🎯 **Range propagates** down the tree  
🎯 **Inorder traversal** must be strictly increasing  
🎯 **Check entire subtrees**, not just children  
🎯 **No duplicates** allowed in BST  
🎯 **O(n) time** to validate  
🎯 **Three valid approaches**: range, inorder recursive, inorder iterative  

Master this problem and you'll understand:
- BST properties deeply
- Range constraint propagation
- Inorder traversal applications
- Recursive vs iterative trade-offs